---
title: Ekstraksi Fitur pada Gambar
---

## Pendahuluan

Dalam bidang *computer vision* dan *image processing*, ekstraksi fitur merupakan tahapan krusial yang bertujuan untuk merepresentasikan informasi visual dari suatu gambar ke dalam bentuk vektor numerik. Representasi numerik ini memungkinkan algoritma *machine learning* untuk memahami dan membedakan pola-pola yang terkandung dalam gambar secara kuantitatif.

Pada notebook ini, dilakukan percobaan ekstraksi fitur terhadap dataset gambar yang terdiri dari dua kelas, yaitu **BERANTAKAN** (ruangan yang tidak tertata rapi) dan **RAPI** (ruangan yang tertata dengan baik). Dataset ini dikumpulkan dari beberapa kelompok mahasiswa melalui platform WhatsApp.

Fitur-fitur yang diekstraksi meliputi:

1. **Fitur Statistik Warna (Color Statistics)**:
   - *Mean Red*, *Mean Green*, *Mean Blue*: Nilai rata-rata intensitas piksel untuk masing-masing kanal warna RGB.
   - *Standard Deviation Red*, *Standard Deviation Green*, *Standard Deviation Blue*: Nilai simpangan baku intensitas piksel untuk masing-masing kanal warna RGB, yang merepresentasikan sebaran atau variasi warna dalam gambar.

2. **Fitur Tekstur GLCM (*Gray-Level Co-occurrence Matrix*)**:
   - *Contrast*: Mengukur variasi intensitas lokal antara piksel tetangga.
   - *Dissimilarity*: Mengukur perbedaan rata-rata intensitas antara piksel tetangga.
   - *Homogeneity*: Mengukur kedekatan distribusi elemen GLCM terhadap diagonal GLCM.
   - *Energy*: Mengukur keseragaman tekstur (juga dikenal sebagai *Angular Second Moment*).
   - *Correlation*: Mengukur korelasi linier antara piksel tetangga.
   - *ASM (Angular Second Moment)*: Kuadrat dari *Energy*, mengukur homogenitas tekstur.

   Setiap properti GLCM dihitung pada 4 arah sudut (0°, 45°, 90°, 135°), sehingga menghasilkan 4 nilai per properti. Untuk merangkum karakteristik tekstur secara keseluruhan tanpa bergantung pada satu arah tertentu, diambil nilai **mean** (rata-rata) dan **std** (simpangan baku) dari ke-4 arah tersebut. Oleh karena itu, setiap properti GLCM menghasilkan dua fitur: `mean` dan `std`.

Notebook ini belum melakukan klasifikasi; fokus utamanya adalah pada proses ekstraksi fitur dan penyimpanan hasilnya dalam format CSV.

## Persiapan

### Import Library

Berikut adalah *library* yang digunakan dalam proses ekstraksi fitur gambar:

- **os** dan **glob**: Untuk navigasi direktori dan pencarian file gambar berdasarkan pola *path*.
- **NumPy (`numpy`)**: Untuk komputasi numerik pada matriks piksel gambar.
- **Pandas (`pandas`)**: Untuk membentuk dan menyimpan tabel hasil ekstraksi fitur.
- **Pillow (`PIL`)**: Untuk membaca dan memproses file gambar.
- **scikit-image (`skimage`)**: Untuk menghitung fitur tekstur GLCM (*Gray-Level Co-occurrence Matrix*).
- **Matplotlib (`matplotlib`)**: Untuk visualisasi sampel gambar dari dataset.

In [ ]:
import os
import glob

import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import graycomatrix, graycoprops
from skimage.color import rgb2gray
from skimage.util import img_as_ubyte
import matplotlib.pyplot as plt

### Konfigurasi Path

Mendefinisikan *path* direktori dataset dan direktori *output* untuk menyimpan hasil ekstraksi.

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
DATASET_DIR = os.path.join(BASE_DIR, "dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

os.makedirs(OUTPUT_DIR, exist_ok=True)

KELAS = ["BERANTAKAN", "RAPI"]

print(f"Dataset directory: {DATASET_DIR}")
print(f"Output directory : {OUTPUT_DIR}")

## Eksplorasi Dataset

### Distribusi Jumlah Gambar per Kelas

Menghitung jumlah gambar yang tersedia di setiap kelas dan kelompok untuk memahami distribusi dataset.

In [ ]:
distribusi = []

for kelas in KELAS:
    kelas_dir = os.path.join(DATASET_DIR, kelas)
    kelompok_dirs = sorted(os.listdir(kelas_dir), key=str.lower)

    for kelompok in kelompok_dirs:
        kelompok_path = os.path.join(kelas_dir, kelompok)
        if not os.path.isdir(kelompok_path):
            continue
        gambar_files = glob.glob(os.path.join(kelompok_path, "*.jpeg")) + \
                       glob.glob(os.path.join(kelompok_path, "*.jpg")) + \
                       glob.glob(os.path.join(kelompok_path, "*.png"))
        distribusi.append({
            "Kelas": kelas,
            "Kelompok": kelompok,
            "Jumlah Gambar": len(gambar_files)
        })

df_distribusi = pd.DataFrame(distribusi)
print(df_distribusi.to_string(index=False))
print(f"\nTotal gambar: {df_distribusi['Jumlah Gambar'].sum()}")
print(f"\nDistribusi per kelas:")
print(df_distribusi.groupby("Kelas")["Jumlah Gambar"].sum())

### Visualisasi Sampel Gambar

Menampilkan beberapa sampel gambar dari masing-masing kelas untuk memberikan gambaran visual terhadap dataset yang digunakan.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, kelas in enumerate(KELAS):
    kelas_dir = os.path.join(DATASET_DIR, kelas)
    all_images = []
    for kelompok in sorted(os.listdir(kelas_dir), key=str.lower):
        kelompok_path = os.path.join(kelas_dir, kelompok)
        if not os.path.isdir(kelompok_path):
            continue
        all_images.extend(
            glob.glob(os.path.join(kelompok_path, "*.jpeg")) +
            glob.glob(os.path.join(kelompok_path, "*.jpg")) +
            glob.glob(os.path.join(kelompok_path, "*.png"))
        )

    np.random.seed(42)
    sampel = np.random.choice(all_images, size=min(4, len(all_images)), replace=False)

    for j, img_path in enumerate(sampel):
        img = Image.open(img_path).convert("RGB")
        axes[i, j].imshow(img)
        axes[i, j].set_title(f"{kelas}", fontsize=11)
        axes[i, j].axis("off")

plt.suptitle("Sampel Gambar dari Setiap Kelas", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Ekstraksi Fitur

### Definisi Fungsi Ekstraksi

Mendefinisikan fungsi untuk mengekstraksi fitur statistik warna (Mean dan Stdev untuk kanal R, G, B) serta fitur tekstur GLCM dari sebuah gambar.

In [ ]:
def ekstraksi_fitur_warna(img_array):
    r, g, b = img_array[:, :, 0], img_array[:, :, 1], img_array[:, :, 2]

    fitur = {
        "mean_r": np.mean(r),
        "mean_g": np.mean(g),
        "mean_b": np.mean(b),
        "std_r": np.std(r),
        "std_g": np.std(g),
        "std_b": np.std(b),
    }

    return fitur

In [ ]:
def ekstraksi_fitur_glcm(img_array):
    gray = rgb2gray(img_array)
    gray_ubyte = img_as_ubyte(gray)

    glcm = graycomatrix(
        gray_ubyte,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256,
        symmetric=True,
        normed=True
    )

    properti = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]

    fitur = {}
    for prop in properti:
        vals = graycoprops(glcm, prop)
        fitur[f"glcm_{prop}_mean"] = np.mean(vals)
        fitur[f"glcm_{prop}_std"] = np.std(vals)

    return fitur

In [ ]:
def ekstraksi_fitur_gambar(img_path):
    img = Image.open(img_path).convert("RGB")
    img_resized = img.resize((256, 256))
    img_array = np.array(img_resized)

    fitur_warna = ekstraksi_fitur_warna(img_array)
    fitur_glcm = ekstraksi_fitur_glcm(img_array)

    fitur = {**fitur_warna, **fitur_glcm}
    return fitur

### Proses Ekstraksi Fitur untuk Seluruh Dataset

Melakukan iterasi terhadap seluruh gambar di setiap kelas dan kelompok, kemudian mengekstraksi fitur dari masing-masing gambar.

In [ ]:
hasil_ekstraksi = []

for kelas in KELAS:
    kelas_dir = os.path.join(DATASET_DIR, kelas)
    kelompok_dirs = sorted(os.listdir(kelas_dir), key=str.lower)

    for kelompok in kelompok_dirs:
        kelompok_path = os.path.join(kelas_dir, kelompok)
        if not os.path.isdir(kelompok_path):
            continue

        gambar_files = sorted(
            glob.glob(os.path.join(kelompok_path, "*.jpeg")) +
            glob.glob(os.path.join(kelompok_path, "*.jpg")) +
            glob.glob(os.path.join(kelompok_path, "*.png"))
        )

        print(f"[*] Memproses {kelas}/{kelompok} ({len(gambar_files)} gambar)")

        for img_path in gambar_files:
            try:
                fitur = ekstraksi_fitur_gambar(img_path)
                fitur["filename"] = os.path.basename(img_path)
                fitur["kelompok"] = kelompok
                fitur["kelas"] = kelas
                hasil_ekstraksi.append(fitur)
            except Exception as e:
                print(f"    [!] Error pada {os.path.basename(img_path)}: {e}")

print(f"\nTotal gambar berhasil diekstraksi: {len(hasil_ekstraksi)}")

## Hasil Ekstraksi

### Pembentukan DataFrame

Mengonversi hasil ekstraksi fitur dari seluruh gambar menjadi *DataFrame* Pandas untuk memudahkan analisis dan penyimpanan.

In [ ]:
kolom_urut = [
    "filename", "kelas", "kelompok",
    "mean_r", "mean_g", "mean_b",
    "std_r", "std_g", "std_b",
    "glcm_contrast_mean", "glcm_contrast_std",
    "glcm_dissimilarity_mean", "glcm_dissimilarity_std",
    "glcm_homogeneity_mean", "glcm_homogeneity_std",
    "glcm_energy_mean", "glcm_energy_std",
    "glcm_correlation_mean", "glcm_correlation_std",
    "glcm_ASM_mean", "glcm_ASM_std"
]

df_fitur = pd.DataFrame(hasil_ekstraksi)[kolom_urut]
print(f"Dimensi DataFrame: {df_fitur.shape}")
print(f"Kolom: {list(df_fitur.columns)}")
df_fitur.head(10)

### Statistik Deskriptif

Menampilkan ringkasan statistik deskriptif dari fitur-fitur yang telah diekstraksi untuk memahami distribusi nilainya.

In [ ]:
fitur_numerik = df_fitur.select_dtypes(include=[np.number])
fitur_numerik.describe().T

### Perbandingan Rata-rata Fitur antar Kelas

Membandingkan nilai rata-rata fitur antara kelas BERANTAKAN dan RAPI untuk melihat perbedaan karakteristik visual yang mungkin dapat digunakan sebagai pembeda dalam klasifikasi.

In [ ]:
perbandingan = df_fitur.groupby("kelas")[fitur_numerik.columns.tolist()].mean().T
perbandingan["selisih"] = abs(perbandingan["BERANTAKAN"] - perbandingan["RAPI"])
perbandingan.sort_values("selisih", ascending=False)

### Visualisasi Perbandingan Fitur Warna

Menampilkan *bar chart* perbandingan rata-rata fitur warna (Mean R, G, B dan Stdev R, G, B) antara kedua kelas.

In [ ]:
fitur_warna_cols = ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b"]
df_warna_mean = df_fitur.groupby("kelas")[fitur_warna_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_warna_mean[["mean_r", "mean_g", "mean_b"]].plot(
    kind="bar", ax=axes[0], color=["red", "green", "blue"], alpha=0.7
)
axes[0].set_title("Rata-rata Mean R, G, B per Kelas")
axes[0].set_ylabel("Nilai Rata-rata")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(title="Kanal")

df_warna_mean[["std_r", "std_g", "std_b"]].plot(
    kind="bar", ax=axes[1], color=["red", "green", "blue"], alpha=0.7
)
axes[1].set_title("Rata-rata Stdev R, G, B per Kelas")
axes[1].set_ylabel("Nilai Rata-rata")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title="Kanal")

plt.tight_layout()
plt.show()

### Visualisasi Perbandingan Fitur GLCM

Menampilkan *bar chart* perbandingan rata-rata fitur tekstur GLCM antara kedua kelas.

In [ ]:
glcm_mean_cols = [
    "glcm_contrast_mean", "glcm_dissimilarity_mean",
    "glcm_homogeneity_mean", "glcm_energy_mean",
    "glcm_correlation_mean", "glcm_ASM_mean"
]

df_glcm_mean = df_fitur.groupby("kelas")[glcm_mean_cols].mean()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for idx, col in enumerate(glcm_mean_cols):
    label = col.replace("glcm_", "").replace("_mean", "").title()
    df_glcm_mean[[col]].plot(kind="bar", ax=axes[idx], legend=False, color=["#2196F3", "#FF9800"])
    axes[idx].set_title(f"GLCM {label}")
    axes[idx].set_ylabel("Nilai Rata-rata")
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=0)

plt.suptitle("Perbandingan Fitur GLCM antar Kelas", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Penyimpanan Hasil

Menyimpan DataFrame hasil ekstraksi fitur ke dalam file CSV di direktori `output/` agar dapat digunakan untuk tahap klasifikasi selanjutnya.

In [ ]:
output_path = os.path.join(OUTPUT_DIR, "hasil_ekstraksi_fitur.csv")
df_fitur.to_csv(output_path, index=False)
print(f"Hasil ekstraksi fitur tersimpan di: {output_path}")
print(f"Jumlah baris: {len(df_fitur)}")
print(f"Jumlah kolom: {len(df_fitur.columns)}")

## Kesimpulan

Pada notebook ini telah dilakukan ekstraksi fitur terhadap seluruh gambar dalam dataset yang terdiri dari dua kelas (BERANTAKAN dan RAPI). Fitur yang diekstraksi meliputi 6 fitur statistik warna (Mean dan Stdev untuk kanal R, G, B) serta 12 fitur tekstur GLCM (Mean dan Stdev dari *Contrast*, *Dissimilarity*, *Homogeneity*, *Energy*, *Correlation*, dan *ASM*), menghasilkan total 18 fitur numerik per gambar.

Hasil ekstraksi fitur telah disimpan dalam format CSV dan siap digunakan untuk tahap klasifikasi pada eksperimen selanjutnya.